In [ ]:
# Ensure notebook runs from repository root and imports resolve
import os
import sys

def find_repo_root(start=None):
    start = start or os.getcwd()
    cur = os.path.abspath(start)
    while True:
        if os.path.exists(os.path.join(cur, 'pyproject.toml')) or os.path.exists(os.path.join(cur, 'README.md')):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            return os.path.abspath(start)
        cur = parent

repo_root = find_repo_root()
print('Setting repo root to', repo_root)
if os.getcwd() != repo_root:
    os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)


# Train LeNet5 on MNIST
Train for 100 epochs using `AdamW` (default params) and batch size 32.
Saves the trained model to /home/lgreen/projects/Online_BS/models/teacher/MNIST.tar and prints final training and validation accuracies.

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from models.LeNet import create_model
from tqdm import tqdm

In [ ]:
# Data transforms and loaders
batch_size = 32
epochs = 20
subset_fraction = 0.1  # use 10% of the training set
transform = transforms.Compose([
    transforms.Resize((32, 32)),  # LeNet expects 32x32
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

data_root = os.path.join(os.getcwd(), 'data')
# full training dataset (we'll split it into train/val based on indices)
train_dataset = datasets.MNIST(root=data_root, train=True, download=True, transform=transform)
val_dataset = datasets.MNIST(root=data_root, train=False, download=True, transform=transform)

# Reproducible seeding
import numpy as _np
import random
seed = 1
_np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Create a class-balanced 10% subset of the training dataset and save indices
labels = _np.array(train_dataset.targets)
indices_10 = []
for c in range(10):
    class_idx = _np.where(labels == c)[0]
    _np.random.shuffle(class_idx)
    k = max(1, int(len(class_idx) * subset_fraction))
    indices_10.extend(class_idx[:k].tolist())

indices_10 = _np.array(indices_10, dtype=_np.int64)
# ensure reproducibility order
indices_10.sort()

# save indices to data/data_utils
idx_save_path = os.path.join(repo_root if 'repo_root' in globals() else os.getcwd(), 'data', 'data_utils', 'mnist_10pct_indices.npy')
os.makedirs(os.path.dirname(idx_save_path), exist_ok=True)
_np.save(idx_save_path, indices_10)

# complementary 90% indices for validation (from the original training set)
all_idx = _np.arange(len(train_dataset))
indices_90 = _np.setdiff1d(all_idx, indices_10)

from torch.utils.data import Subset
train_subset = Subset(train_dataset, indices_10.tolist())
val_subset = Subset(train_dataset, indices_90.tolist())

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)


In [ ]:
# Build model, optimizer, loss
# Enforce use of CUDA device 1
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available; training requires cuda:1')
ngpu = torch.cuda.device_count()
if ngpu <= 1:
    raise RuntimeError(f'CUDA device 1 not available; found {ngpu} GPU(s)')

device = torch.device('cuda:0')
print('Using device', device)

model = create_model(in_channels=1, num_classes=10).to(device)
optimizer = torch.optim.AdamW(model.parameters())
criterion = nn.CrossEntropyLoss()

# Save path for best model (lowest validation loss)
save_path = os.path.join(repo_root, 'models', 'teacher', 'MNIST.tar') if 'repo_root' in globals() else '/home/lgreen/projects/Online_BS/models/teacher/MNIST.tar'
os.makedirs(os.path.dirname(save_path), exist_ok=True)

best_val_loss = float('inf')
best_epoch = -1

# Training loop with tqdm progress bars
for epoch in range(1, epochs + 1):
    model.train()
    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", leave=False)
    for xb, yb in loop:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        loop.set_postfix(loss=loss.item())

    # After each epoch: evaluate on validation set (90% holdout) and track loss
    model.eval()
    total_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for xb_val, yb_val in val_loader:
            xb_val = xb_val.to(device, non_blocking=True)
            yb_val = yb_val.to(device, non_blocking=True)
            out_val = model(xb_val)
            loss_val = criterion(out_val, yb_val)
            total_loss += loss_val.item() * yb_val.size(0)
            total_samples += yb_val.size(0)
    val_loss = total_loss / total_samples if total_samples > 0 else float('inf')

    print(f'Epoch {epoch}: validation loss = {val_loss:.6f}')

    # Save model if validation loss improved
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), save_path)
        print(f'New best model (epoch {epoch}) saved to: {save_path}')

# After training: evaluate on train and validation sets (final metrics)
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            out = model(xb)
            preds = out.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / total

train_acc = evaluate(train_loader)
val_acc = evaluate(val_loader)

print(f'Best validation loss {best_val_loss:.6f} at epoch {best_epoch}')
print(f'Final training accuracy: {train_acc * 100:.2f}%')
print(f'Final validation accuracy: {val_acc * 100:.2f}%')
print(f'Best model saved to: {save_path}')
